# 🚗 Driver Drowsiness & Distraction Detection System
## Notebook 1: Dataset Setup & Preprocessing

### Datasets used:
| Task | Dataset | Source | Size |
|------|---------|--------|------|
| Eye closure (EAR) | MRL Eye Dataset | Kaggle | ~84k images |
| Yawning (MAR) | Yawning Detection Dataset | Kaggle | ~3k images |
| Head Pose | 300W-LP / BIWI | Kaggle/HuggingFace | ~60k images |
| Phone detection | COCO (person+phone) / Custom | Roboflow | ~3k images |
| End-to-end drowsy | NTHU-DDD / CEW | Kaggle | ~2k videos |


In [ ]:
# ==============================================================
# CELL 1 — Install dependencies
# ==============================================================
!pip install -q opencv-python mediapipe ultralytics numpy scipy \
    matplotlib seaborn pandas scikit-learn Pillow pygame tqdm \
    imutils gdown kaggle

In [ ]:
# ==============================================================
# CELL 2 — Mount Google Drive (to save datasets persistently)
# ==============================================================
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = '/content/drive/MyDrive/DriverMonitoring'
os.makedirs(f'{BASE}/data/raw', exist_ok=True)
os.makedirs(f'{BASE}/data/processed', exist_ok=True)
os.makedirs(f'{BASE}/models', exist_ok=True)
os.makedirs(f'{BASE}/results', exist_ok=True)
print('Directory structure ready:', BASE)

In [ ]:
# ==============================================================
# CELL 3 — Download MRL Eye Dataset (Eye closure / drowsiness)
# Dataset: https://www.kaggle.com/datasets/imadeddinedjerarda/mrl-eye-dataset
# ==============================================================
import gdown, zipfile, os

# Option A: via Kaggle API (set kaggle.json first)
# !kaggle datasets download -d imadeddinedjerarda/mrl-eye-dataset -p {BASE}/data/raw/

# Option B: Direct gdown from shared link
# Replace with your own Kaggle download URL
MRL_URL = 'https://www.kaggle.com/api/v1/datasets/download/imadeddinedjerarda/mrl-eye-dataset'
print('Visit https://www.kaggle.com/datasets/imadeddinedjerarda/mrl-eye-dataset')
print('Download manually and upload to:', f'{BASE}/data/raw/mrl_eye.zip')
print('OR use Kaggle API by placing kaggle.json in /root/.kaggle/')

In [ ]:
# ==============================================================
# CELL 4 — Kaggle API setup (run once)
# ==============================================================
from google.colab import files
print('Upload your kaggle.json file:')
uploaded = files.upload()

import os
os.makedirs('/root/.kaggle', exist_ok=True)
os.rename('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('Kaggle API configured.')

In [ ]:
# ==============================================================
# CELL 5 — Download all datasets via Kaggle CLI
# ==============================================================
import os
RAW = f'{BASE}/data/raw'

datasets = {
    'mrl_eye':    'imadeddinedjerarda/mrl-eye-dataset',
    'yawn':       'davidvazquezcic/yawn-eye-dataset-new',
    'drowsy_drv': 'ismailnasri20/driver-drowsiness-dataset',
}

for folder, slug in datasets.items():
    dest = f'{RAW}/{folder}'
    os.makedirs(dest, exist_ok=True)
    print(f'Downloading {slug}...')
    os.system(f'kaggle datasets download -d {slug} -p {dest} --unzip')
    print(f'  -> Done: {dest}')

print('\nAll datasets downloaded!')

In [ ]:
# ==============================================================
# CELL 6 — Dataset exploration & statistics
# ==============================================================
import os
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

def explore_dataset(root_path, name, max_classes=8):
    root = Path(root_path)
    classes = [d for d in root.iterdir() if d.is_dir()]
    stats = {c.name: len(list(c.glob('*.jpg')) + list(c.glob('*.png'))) for c in classes}
    
    print(f'\n=== {name} ===' )
    print(f'  Classes: {len(classes)}')
    print(f'  Total images: {sum(stats.values())}')
    for cls, count in list(stats.items())[:max_classes]:
        bar = '█' * (count // 200)
        print(f'  {cls:20s}: {count:5d}  {bar}')
    return stats

# Adjust paths after downloading
# explore_dataset(f'{RAW}/mrl_eye', 'MRL Eye Dataset')
# explore_dataset(f'{RAW}/yawn', 'Yawn Dataset')
print('Run after downloading datasets.')

In [ ]:
# ==============================================================
# CELL 7 — Preprocessing pipeline: Eye crop & normalization
# ==============================================================
import cv2
import numpy as np
import mediapipe as mp
from pathlib import Path
from tqdm import tqdm

mp_face_mesh = mp.solutions.face_mesh

# MediaPipe landmark indices for eyes
LEFT_EYE_IDX  = [362, 382, 381, 380, 374, 373, 390, 249, 263, 466, 388, 387, 386, 385, 384, 398]
RIGHT_EYE_IDX = [33, 7, 163, 144, 145, 153, 154, 155, 133, 173, 157, 158, 159, 160, 161, 246]
MOUTH_IDX     = [61, 291, 39, 181, 0, 17, 269, 405, 321, 375, 405, 91]

def preprocess_face_image(img_path, target_size=(224, 224)):
    """
    Preprocess a face image:
    1. Detect face with MediaPipe
    2. Crop and align
    3. Normalize to [0,1]
    Returns: processed image or None if no face detected
    """
    img = cv2.imread(str(img_path))
    if img is None:
        return None
    
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    
    with mp_face_mesh.FaceMesh(
        static_image_mode=True,
        max_num_faces=1,
        refine_landmarks=True,
        min_detection_confidence=0.5
    ) as face_mesh:
        results = face_mesh.process(rgb)
        if not results.multi_face_landmarks:
            # Fallback: resize without landmark crop
            img_resized = cv2.resize(img, target_size)
            return img_resized.astype(np.float32) / 255.0
        
        lm = results.multi_face_landmarks[0].landmark
        xs = [l.x * w for l in lm]
        ys = [l.y * h for l in lm]
        
        # Tight face bounding box with 20% padding
        pad = 0.20
        x1 = max(0, int(min(xs) - pad * w))
        y1 = max(0, int(min(ys) - pad * h))
        x2 = min(w, int(max(xs) + pad * w))
        y2 = min(h, int(max(ys) + pad * h))
        
        face_crop = img[y1:y2, x1:x2]
        face_resized = cv2.resize(face_crop, target_size)
        return face_resized.astype(np.float32) / 255.0

print('Preprocessing functions defined.')

In [ ]:
# ==============================================================
# CELL 8 — Data augmentation pipeline
# ==============================================================
import cv2
import numpy as np
import random

def augment_image(img, seed=None):
    """
    Realistic in-car augmentations:
    - Random brightness (simulate day/night/tunnel)
    - Random horizontal flip
    - Small rotation (head tilt)
    - Gaussian noise (camera noise)
    - Gaussian blur (focus variation)
    """
    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)
    
    aug = img.copy()
    
    # 1. Brightness/contrast jitter
    if random.random() < 0.6:
        alpha = random.uniform(0.6, 1.4)  # contrast
        beta  = random.randint(-40, 40)   # brightness
        aug = np.clip(aug * alpha + beta, 0, 255).astype(np.uint8)
    
    # 2. Horizontal flip (mirror the driver)
    if random.random() < 0.5:
        aug = cv2.flip(aug, 1)
    
    # 3. Rotation (-15 to +15 degrees)
    if random.random() < 0.5:
        h, w = aug.shape[:2]
        angle = random.uniform(-15, 15)
        M = cv2.getRotationMatrix2D((w//2, h//2), angle, 1.0)
        aug = cv2.warpAffine(aug, M, (w, h), borderMode=cv2.BORDER_REFLECT)
    
    # 4. Gaussian noise
    if random.random() < 0.4:
        noise = np.random.normal(0, random.uniform(2, 8), aug.shape).astype(np.int16)
        aug = np.clip(aug.astype(np.int16) + noise, 0, 255).astype(np.uint8)
    
    # 5. Gaussian blur
    if random.random() < 0.3:
        k = random.choice([3, 5])
        aug = cv2.GaussianBlur(aug, (k, k), 0)
    
    # 6. Occlusion patch (simulate glasses/hand)
    if random.random() < 0.2:
        h, w = aug.shape[:2]
        px, py = random.randint(0, w-40), random.randint(0, h-20)
        pw, ph = random.randint(20, 60), random.randint(10, 30)
        aug[py:py+ph, px:px+pw] = random.randint(0, 50)
    
    return aug

print('Augmentation pipeline defined.')

In [ ]:
# ==============================================================
# CELL 9 — Train/Val/Test split
# ==============================================================
import os
import shutil
import random
from pathlib import Path
from tqdm import tqdm

def split_dataset(src_root, dst_root, splits=(0.70, 0.15, 0.15), seed=42):
    """
    Splits a folder-per-class dataset into train/val/test.
    Maintains class balance across splits.
    """
    random.seed(seed)
    src = Path(src_root)
    dst = Path(dst_root)
    
    for split in ['train', 'val', 'test']:
        (dst / split).mkdir(parents=True, exist_ok=True)
    
    stats = {}
    for cls_dir in sorted(src.iterdir()):
        if not cls_dir.is_dir():
            continue
        images = list(cls_dir.glob('*.jpg')) + list(cls_dir.glob('*.png'))
        random.shuffle(images)
        
        n = len(images)
        n_train = int(n * splits[0])
        n_val   = int(n * splits[1])
        
        partitions = {
            'train': images[:n_train],
            'val':   images[n_train:n_train+n_val],
            'test':  images[n_train+n_val:]
        }
        
        for split, imgs in partitions.items():
            split_cls_dir = dst / split / cls_dir.name
            split_cls_dir.mkdir(exist_ok=True)
            for img in imgs:
                shutil.copy2(img, split_cls_dir / img.name)
        
        stats[cls_dir.name] = {'total': n, 'train': len(partitions['train']),
                                'val': len(partitions['val']), 'test': len(partitions['test'])}
        print(f'  {cls_dir.name:20s}: train={stats[cls_dir.name]["train"]:4d} val={stats[cls_dir.name]["val"]:4d} test={stats[cls_dir.name]["test"]:4d}')
    
    return stats

# Example usage (run after dataset download):
# stats = split_dataset(f'{RAW}/yawn', f'{BASE}/data/processed/yawn_split')
print('Split function defined. Run after dataset download.')

In [ ]:
# ==============================================================
# CELL 10 — Visualize sample augmentations
# ==============================================================
import matplotlib.pyplot as plt
import cv2

def visualize_augmentations(img_path, n_aug=6):
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    fig, axes = plt.subplots(1, n_aug+1, figsize=(18, 3))
    axes[0].imshow(img)
    axes[0].set_title('Original', fontsize=10)
    axes[0].axis('off')
    
    for i in range(n_aug):
        aug = augment_image(img.copy(), seed=i*7)
        axes[i+1].imshow(aug)
        axes[i+1].set_title(f'Aug {i+1}', fontsize=10)
        axes[i+1].axis('off')
    
    plt.suptitle('Data Augmentation Samples', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{BASE}/results/augmentation_samples.png', dpi=150, bbox_inches='tight')
    plt.show()

# visualize_augmentations('path/to/sample.jpg')
print('Visualization function ready.')